In [59]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from tensorflow.keras.layers import Input,Dense
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping

In [60]:
df = pd.read_csv('student_mental_health_burnout.csv')

In [61]:
df.head()

,student_id,age,gender,course,year,daily_study_hours,daily_sleep_hours,screen_time_hours,stress_level,anxiety_score,depression_score,academic_pressure_score,financial_stress_score,social_support_score,physical_activity_hours,sleep_quality,attendance_percentage,cgpa,internet_quality,burnout_level
0,100001,23,Male,BTech,1st,4.3,6.8,6.1,High,10,3,4,2,6,1.8,Average,66.5,9.63,Good,High
1,100002,20,Male,BTech,3rd,1.4,4.7,3.0,High,2,10,8,5,9,1.9,Poor,55.8,6.04,Poor,Low
2,100003,24,Female,BCA,4th,3.7,4.8,1.5,Low,2,7,8,6,3,0.8,Good,85.0,8.31,Good,High
3,100004,21,Male,BSc,4th,1.6,6.7,7.0,High,3,3,4,9,9,0.7,Poor,89.1,5.95,Good,High
4,100005,23,Other,BSc,4th,2.0,6.7,5.4,High,7,7,6,4,4,1.7,Good,58.7,8.51,Good,Low


In [62]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 20 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   student_id               150000 non-null  int64  
 1   age                      150000 non-null  int64  
 2   gender                   150000 non-null  object 
 3   course                   150000 non-null  object 
 4   year                     150000 non-null  object 
 5   daily_study_hours        150000 non-null  float64
 6   daily_sleep_hours        150000 non-null  float64
 7   screen_time_hours        150000 non-null  float64
 8   stress_level             150000 non-null  object 
 9   anxiety_score            150000 non-null  int64  
 10  depression_score         150000 non-null  int64  
 11  academic_pressure_score  150000 non-null  int64  
 12  financial_stress_score   150000 non-null  int64  
 13  social_support_score     150000 non-null  int64  
 14  phys

In [63]:
df.drop(columns = 'student_id' , inplace = True)

In [74]:
for col in df.select_dtypes(include='object').columns:
    print(f"--- {col} column ---")
    print(df[col].unique(),df[col].nunique())

--- gender column ---
['Male' 'Female' 'Other'] 3
--- course column ---
['BTech' 'BCA' 'BSc' 'MBA' 'MCA' 'BBA'] 6
--- year column ---
['1st' '3rd' '4th' '2nd'] 4
--- stress_level column ---
['High' 'Low' 'Medium'] 3
--- sleep_quality column ---
['Average' 'Poor' 'Good'] 3
--- internet_quality column ---
['Good' 'Poor' 'Average'] 3
--- burnout_level column ---
['High' 'Low' 'Medium'] 3


In [65]:
cat_cols =  df.select_dtypes(include=['object']).columns.tolist()
if 'burnout_level' in cat_cols:
    cat_cols.remove('burnout_level')

In [66]:
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

In [67]:
X = df.drop(columns = 'burnout_level')
y = df['burnout_level']

In [68]:
X_train , X_test , y_train , y_test = train_test_split(
    X , y , test_size = 0.2 , random_state = 42
)

In [69]:
num_pip = Pipeline([
    ('scaler', StandardScaler())
])

In [70]:
cat_pip = Pipeline([
    ('one', OneHotEncoder(handle_unknown = 'ignore'))
])

In [71]:
data = ColumnTransformer([
    ('cat', cat_pip , cat_cols),
    ('num', num_pip , num_cols)
])

In [72]:
X_train_ready = data.fit_transform(X_train)
X_test_ready = data.transform(X_test)

In [103]:
model = Sequential([
    Input(X_train_ready.shape[1]),

    Dense(64, activation = 'relu'),
    Dense(32, activation = 'relu'),

    Dense(3, activation = 'softmax')
])

In [104]:
model.compile(
    optimizer = 'adam',
    loss = 'sparse_categorical_crossentropy',
    metrics = ['accuracy']
)

In [105]:
early_stop = EarlyStopping(patience = 10,monitor = 'val_loss', mode = 'min', restore_best_weights = True)

In [106]:
model.fit(
    X_train_ready,y_train,
    epochs = 100,
    batch_size = 32,
    validation_split = 0.2,
    callbacks = [early_stop],
    verbose = 1
)

Epoch 1/100
3000/3000 [==============================] - 1s 341us/step - loss: 1.1010 - accuracy: 0.3326 - val_loss: 1.0992 - val_accuracy: 0.3265
Epoch 2/100
3000/3000 [==============================] - 1s 326us/step - loss: 1.0986 - accuracy: 0.3376 - val_loss: 1.1002 - val_accuracy: 0.3257
Epoch 3/100
3000/3000 [==============================] - 1s 318us/step - loss: 1.0982 - accuracy: 0.3415 - val_loss: 1.0993 - val_accuracy: 0.3326
Epoch 4/100
3000/3000 [==============================] - 1s 317us/step - loss: 1.0977 - accuracy: 0.3452 - val_loss: 1.1001 - val_accuracy: 0.3350
Epoch 5/100
3000/3000 [==============================] - 1s 326us/step - loss: 1.0964 - accuracy: 0.3505 - val_loss: 1.1006 - val_accuracy: 0.3280
Epoch 6/100
3000/3000 [==============================] - 1s 319us/step - loss: 1.0950 - accuracy: 0.3560 - val_loss: 1.1021 - val_accuracy: 0.3303
Epoch 7/100
3000/3000 [==============================] - 1s 317us/step - loss: 1.0933 - accuracy: 0.3611 - val_loss: 1